### Tools

#### Models can request to call tools that perform tasks such as fetching data from a database, searching web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.


In [ ]:
import os
from langchain_groq import ChatGroq

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')


model=ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")
response

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    return f"The weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response=model_with_tools.invoke("What is the weather like in New York?")
for tool_call in response.tool_calls:
    #View tools calls made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")
    

### Tool Execution Loops

In [ ]:
# Step 1 : Model generates tool calls
messages = [
    {"role": "user", "content": "What is the weather like in New York?"}
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


In [ ]:
messages